In [27]:
import json, os, re
from collections import defaultdict, OrderedDict
import pandas as pd
from IPython.display import display, HTML

# ========= CONFIG =========
JSON_FILES = [
    "predictions/NER_G.json",
    "predictions/NER_S.json",
    "predictions/NER_SG.json",
    "predictions/gold_only_pred.json",
    "predictions/silver_only_pred.json",
    "predictions/silver_to_gold_pred.json",
]

# Merged 5-entity schema (output order)
ENTITIES = OrderedDict([
    ("Recipient", "Recipient"),
    ("Authority", "Authority"),
    ("Legal Object", "Legal Object"),
    ("Legal Action", "Legal Action"),
    ("Legal Basis", "Legal Basis"),
])

# Optional mapping from the first 10 chars of id to readable names
AUTHORITY_MAP = {
    "kansspelau": "KSA",
    "open.overh": "Rijksoverheid",
    "puc.overhe": "ANVS",
    "www.acm.nl": "ACM",
    "www.dnb.nl": "DNB",
    "www.evenem": "Gemeente Rotterdam",
    "www.provin": "Provincie Drenthe",
}
def map_authority(auth_id10: str) -> str:
    return AUTHORITY_MAP.get(auth_id10, auth_id10)

# ========= TAG NORMALIZATION / MAPPING =========
# Numeric tag schemas merged to 5 entities
NUMERIC_GROUPS = {
    1: "Recipient", 2: "Recipient",
    3: "Authority", 4: "Authority",
    5: "Legal Object", 6: "Legal Object",
    7: "Legal Action", 8: "Legal Action",
    9: "Legal Basis", 10: "Legal Basis",
}

BIO_PREFIX_RE = re.compile(r"^(?:[BIOUESL]-)+", re.IGNORECASE)   # strip BIO-like prefixes
NON_ALNUM_RE = re.compile(r"[^a-z0-9]+", re.IGNORECASE)           # collapse to [a-z0-9]

def _canon(s: str) -> str:
    """Strip BIO, lowercase, remove non-alnum, normalize a few diacritics."""
    s = BIO_PREFIX_RE.sub("", s.strip()).lower()
    s = (s
         .replace("é","e").replace("è","e").replace("ë","e")
         .replace("ï","i").replace("à","a").replace("ä","a"))
    return NON_ALNUM_RE.sub("", s)

# Canonical aliases to 5 entities (keys must be canonical like 'wettelijkegrondslag')
STRING_ALIASES = {
    # Recipient
    "ontvanger": "Recipient", "recipient": "Recipient",

    # Authority
    "besluitvormendorgaan": "Authority", "orgaan": "Authority", "authority": "Authority",

    # Legal Object
    "rechtsobject": "Legal Object", "legalobject": "Legal Object", "object": "Legal Object",

    # Legal Action
    "rechtshandeling": "Legal Action", "legalaction": "Legal Action", "handeling": "Legal Action",
    "wettelijkeactie": "Legal Action",    # handles Wettelijke_actie (and BIO forms)

    # Legal Basis
    "grondslag": "Legal Basis", "rechtsgrond": "Legal Basis", "legalbasis": "Legal Basis",
    "basis": "Legal Basis", "wettelijkegrondslag": "Legal Basis",  # handles Wettelijke_grondslag
}

def tag_to_entity(tag):
    """Map a raw tag (int/str/BIO) to merged entity or None (outside)."""
    if tag is None:
        return None
    # int label schema
    if isinstance(tag, int):
        return NUMERIC_GROUPS.get(tag)
    # string label schema (O, numeric strings, BIO, text)
    if isinstance(tag, str):
        s = tag.strip()
        if not s or s.upper() == "O":
            return None
        try:
            n = int(s)
            return NUMERIC_GROUPS.get(n)
        except ValueError:
            pass
        key = _canon(s)
        if not key:
            return None
        ent = STRING_ALIASES.get(key)
        if ent:
            return ent
        # light fallback heuristics
        if key.endswith("grondslag") or key.endswith("rechtsgrond"):
            return "Legal Basis"
        if key.endswith("actie") or key.endswith("handeling") or key.endswith("handelingen"):
            return "Legal Action"
    return None

def get_gold_and_pred(item):
    """Robust field extraction across schemas."""
    gold = item.get("ner_tags") or item.get("labels")
    if gold is None:
        raise KeyError(f"No gold tags in item id={item.get('id')}")
    if "pred_tags" in item:
        pred = item["pred_tags"]
    elif "predicted_ner_tags" in item:
        pred = item["predicted_ner_tags"]
    else:
        raise KeyError(f"No prediction tags in item id={item.get('id')}")
    return gold, pred

# ========= SPAN RECONSTRUCTION (strict span-level) =========
def sequence_to_entity_labels(seq):
    """Token-wise normalized entity labels or None for outside."""
    return [tag_to_entity(t) for t in seq]

def spans_from_entity_labels(labels):
    """
    Build contiguous spans per entity: {entity -> set((start,end))}, end-exclusive.
    Consecutive tokens with the same entity become ONE span (BIO ignored).
    """
    spans = {e: set() for e in ENTITIES.keys()}
    current_ent, start = None, None
    for i, ent in enumerate(labels + [None]):  # sentinel to flush last
        if ent != current_ent:
            if current_ent is not None:
                spans[current_ent].add((start, i))
            current_ent, start = ent, (i if ent is not None else None)
    return spans

# ========= STRICT METRICS =========
def safe_div(n, d): 
    return n / d if d else 0.0

def prf1_from_sets_strict(gold_set, pred_set):
    """
    Strict span scoring: TP if and only if (start,end) matches exactly.
    gold_set/pred_set members are (item_idx, start, end).
    """
    tp = len(gold_set & pred_set)
    fp = len(pred_set - gold_set)
    fn = len(gold_set - pred_set)
    P = safe_div(tp, tp + fp)
    R = safe_div(tp, tp + fn)
    F1 = safe_div(2 * P * R, P + R)
    return P, R, F1, len(gold_set), tp, fp, fn  # include gold count and components

def micro_from_span_sets_strict(gold_dict, pred_dict):
    """Micro-averaged P/R/F1 across all entities (strict)."""
    TP = FP = FN = 0
    for ent in ENTITIES.keys():
        g = gold_dict.get(ent, set())
        p = pred_dict.get(ent, set())
        TP += len(g & p)
        FP += len(p - g)
        FN += len(g - p)
    microP = safe_div(TP, TP + FP)
    microR = safe_div(TP, TP + FN)
    microF1 = safe_div(2 * microP * microR, microP + microR)
    return microP, microR, microF1, TP, FP, FN

# ========= PROCESS ONE JSON (strict span-level with micro) =========
def process_json_span_level_with_micro(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # Per-authority span accumulators
    auth_gold_spans = defaultdict(lambda: {e: set() for e in ENTITIES.keys()})
    auth_pred_spans = defaultdict(lambda: {e: set() for e in ENTITIES.keys()})
    # Global accumulators
    global_gold_spans = {e: set() for e in ENTITIES.keys()}
    global_pred_spans = {e: set() for e in ENTITIES.keys()}

    for idx, item in enumerate(data):
        try:
            gold, pred = get_gold_and_pred(item)
        except KeyError:
            continue
        n = min(len(gold), len(pred))
        if n == 0:
            continue

        gold_labels = sequence_to_entity_labels(gold[:n])
        pred_labels = sequence_to_entity_labels(pred[:n])

        gold_spans = spans_from_entity_labels(gold_labels)
        pred_spans = spans_from_entity_labels(pred_labels)

        auth_key = map_authority(str(item.get("id", ""))[:10])

        for ent in ENTITIES.keys():
            # tag spans with item index to keep them unique globally
            gtagged = {(idx, a, b) for (a, b) in gold_spans[ent]}
            ptagged = {(idx, a, b) for (a, b) in pred_spans[ent]}
            auth_gold_spans[auth_key][ent].update(gtagged)
            auth_pred_spans[auth_key][ent].update(ptagged)
            global_gold_spans[ent].update(gtagged)
            global_pred_spans[ent].update(ptagged)

    # ---- Per-authority table (per-entity + micro) ----
    rows = []
    for auth in auth_gold_spans.keys() | auth_pred_spans.keys():
        row = {"Authority": auth}
        for ent in ENTITIES.keys():
            P, R, F1, cnt, *_ = prf1_from_sets_strict(auth_gold_spans[auth][ent], auth_pred_spans[auth][ent])
            row[f"{ent} P"] = round(P, 3)
            row[f"{ent} R"] = round(R, 3)
            row[f"{ent} F1"] = round(F1, 3)
            row[f"{ent} #"] = cnt

        microP, microR, microF1, *_ = micro_from_span_sets_strict(auth_gold_spans[auth], auth_pred_spans[auth])
        row["Micro-P"]  = round(microP, 3)
        row["Micro-R"]  = round(microR, 3)
        row["Micro-F1"] = round(microF1, 3)
        rows.append(row)

    columns_auth = ["Authority"]
    for ent in ENTITIES.keys():
        columns_auth += [f"{ent} P", f"{ent} R", f"{ent} F1", f"{ent} #"]
    columns_auth += ["Micro-P", "Micro-R", "Micro-F1"]

    df_authority = pd.DataFrame(rows)
    for col in columns_auth:
        if col not in df_authority.columns:
            df_authority[col] = float("nan")
    df_authority = df_authority[columns_auth].sort_values("Micro-F1", ascending=False).reset_index(drop=True)

    model_name = os.path.splitext(os.path.basename(path))[0]
    df_authority.insert(0, "Model", model_name)

    # ---- Overall (not divided by authority): per-entity + overall micro row ----
    rows_overall = []
    for ent in ENTITIES.keys():
        P, R, F1, cnt, *_ = prf1_from_sets_strict(global_gold_spans[ent], global_pred_spans[ent])
        rows_overall.append({"Entity": ent, "P": round(P, 3), "R": round(R, 3), "F1": round(F1, 3), "#": cnt})

    o_microP, o_microR, o_microF1, *_ = micro_from_span_sets_strict(global_gold_spans, global_pred_spans)
    df_overall = pd.DataFrame(rows_overall)
    df_overall = pd.concat(
        [df_overall,
         pd.DataFrame([{"Entity": "Micro (all entities)", "P": round(o_microP,3),
                        "R": round(o_microR,3), "F1": round(o_microF1,3), "#": ""}])],
        ignore_index=True
    )
    df_overall.insert(0, "Model", model_name)

    return df_authority, df_overall

# ========= LOOP & DISPLAY =========
all_authority_tables, all_overall_tables = [], []

for path in JSON_FILES:
    if not os.path.exists(path):
        print(f"⚠️ File not found: {path}")
        continue

    df_auth, df_overall = process_json_span_level_with_micro(path)
    all_authority_tables.append(df_auth)
    all_overall_tables.append(df_overall)

    display(HTML(f"<h3>Per-authority results (STRICT span-level, Micro-averaged) — {df_auth['Model'].iloc[0] if not df_auth.empty else os.path.basename(path)}</h3>"))
    display(df_auth.style.format(precision=3).set_table_styles(
        [{'selector': 'th', 'props': [('text-align', 'center')]},
         {'selector': 'td', 'props': [('text-align', 'center')]}]
    ))

    display(HTML(f"<h4>Overall per-entity + Micro row (STRICT span-level) — {df_overall['Model'].iloc[0]}</h4>"))
    display(df_overall.style.format(precision=3).set_table_styles(
        [{'selector': 'th', 'props': [('text-align', 'center')]},
         {'selector': 'td', 'props': [('text-align', 'center')]}]
    ))

# ========= COMBINED SUMMARIES (optional) =========
if all_authority_tables:
    combined_auth = pd.concat(all_authority_tables, ignore_index=True)
    display(HTML("<h2>Combined summary — per authority (STRICT spans, Micro)</h2>"))
    display(combined_auth.style.format(precision=3))

if all_overall_tables:
    combined_overall = pd.concat(all_overall_tables, ignore_index=True)
    display(HTML("<h2>Combined summary — overall per-entity + Micro row (STRICT spans)</h2>"))
    display(combined_overall.style.format(precision=3))


,Model,Authority,Recipient P,Recipient R,Recipient F1,Recipient #,Authority P,Authority R,Authority F1,Authority #,Legal Object P,Legal Object R,Legal Object F1,Legal Object #,Legal Action P,Legal Action R,Legal Action F1,Legal Action #,Legal Basis P,Legal Basis R,Legal Basis F1,Legal Basis #,Micro-P,Micro-R,Micro-F1
0,NER_G,Gemeente Rotterdam,1.000,1.000,1.000,5,1.000,1.000,1.000,5,1.000,1.000,1.000,5,1.000,1.000,1.000,5,1.000,1.000,1.000,5,1.000,1.000,1.000
1,NER_G,Rijksoverheid,1.000,0.875,0.933,8,0.909,1.000,0.952,10,0.917,0.846,0.880,13,1.000,0.923,0.960,13,0.750,1.000,0.857,3,0.935,0.915,0.925
2,NER_G,DNB,0.833,1.000,0.909,5,1.000,0.909,0.952,11,1.000,0.833,0.909,6,0.833,0.833,0.833,6,1.000,1.000,1.000,4,0.935,0.906,0.921
3,NER_G,Provincie Drenthe,1.000,1.000,1.000,7,1.000,0.900,0.947,10,0.875,0.700,0.778,10,0.900,0.900,0.900,10,0.000,0.000,0.000,1,0.941,0.842,0.889
4,NER_G,ANVS,0.579,0.647,0.611,17,0.000,0.000,0.000,0,1.000,1.000,1.000,21,0.955,1.000,0.977,21,0.000,0.000,0.000,0,0.841,0.898,0.869
5,NER_G,KSA,0.667,1.000,0.800,4,0.556,0.714,0.625,7,1.000,0.750,0.857,8,1.000,0.750,0.857,8,1.000,1.000,1.000,6,0.818,0.818,0.818
6,NER_G,ACM,0.750,0.923,0.828,13,0.700,0.875,0.778,24,1.000,0.789,0.882,19,1.000,0.789,0.882,19,0.579,0.846,0.688,13,0.779,0.841,0.809


,Model,Entity,P,R,F1,#
0,NER_G,Recipient,0.773,0.864,0.816,59
1,NER_G,Authority,0.800,0.896,0.845,67
2,NER_G,Legal Object,0.972,0.854,0.909,82
3,NER_G,Legal Action,0.961,0.890,0.924,82
4,NER_G,Legal Basis,0.763,0.906,0.829,32
5,NER_G,Micro (all entities),0.865,0.879,0.872,


,Model,Authority,Recipient P,Recipient R,Recipient F1,Recipient #,Authority P,Authority R,Authority F1,Authority #,Legal Object P,Legal Object R,Legal Object F1,Legal Object #,Legal Action P,Legal Action R,Legal Action F1,Legal Action #,Legal Basis P,Legal Basis R,Legal Basis F1,Legal Basis #,Micro-P,Micro-R,Micro-F1
0,NER_S,Gemeente Rotterdam,0.500,1.000,0.667,5,0.833,1.000,0.909,5,0.833,1.000,0.909,5,1.000,1.000,1.000,5,1.000,0.800,0.889,5,0.774,0.960,0.857
1,NER_S,Provincie Drenthe,1.000,1.000,1.000,7,0.909,1.000,0.952,10,0.500,0.900,0.643,10,0.474,0.900,0.621,10,0.000,0.000,0.000,1,0.636,0.921,0.753
2,NER_S,Rijksoverheid,0.583,0.875,0.700,8,0.833,1.000,0.909,10,0.647,0.846,0.733,13,0.182,0.154,0.167,13,0.000,0.000,0.000,3,0.566,0.638,0.600
3,NER_S,ANVS,0.000,0.000,0.000,17,0.000,0.000,0.000,0,0.955,1.000,0.977,21,0.750,0.429,0.545,21,0.000,0.000,0.000,0,0.600,0.508,0.550
4,NER_S,ACM,0.000,0.000,0.000,13,0.517,0.625,0.566,24,0.347,0.895,0.500,19,0.419,0.684,0.520,19,0.000,0.000,0.000,13,0.308,0.511,0.385
5,NER_S,KSA,0.000,0.000,0.000,4,0.312,0.714,0.435,7,0.292,0.875,0.438,8,0.318,0.875,0.467,8,0.667,0.333,0.444,6,0.276,0.636,0.385
6,NER_S,DNB,0.333,0.200,0.250,5,0.524,1.000,0.688,11,0.000,0.000,0.000,6,0.227,0.833,0.357,6,0.400,0.500,0.444,4,0.241,0.594,0.342


,Model,Entity,P,R,F1,#
0,NER_S,Recipient,0.215,0.339,0.263,59
1,NER_S,Authority,0.583,0.836,0.687,67
2,NER_S,Legal Object,0.427,0.854,0.569,82
3,NER_S,Legal Action,0.410,0.610,0.490,82
4,NER_S,Legal Basis,0.533,0.250,0.340,32
5,NER_S,Micro (all entities),0.416,0.634,0.502,


,Model,Authority,Recipient P,Recipient R,Recipient F1,Recipient #,Authority P,Authority R,Authority F1,Authority #,Legal Object P,Legal Object R,Legal Object F1,Legal Object #,Legal Action P,Legal Action R,Legal Action F1,Legal Action #,Legal Basis P,Legal Basis R,Legal Basis F1,Legal Basis #,Micro-P,Micro-R,Micro-F1
0,NER_SG,Gemeente Rotterdam,1.000,1.000,1.000,5,1.000,1.000,1.000,5,1.000,1.000,1.000,5,1.000,1.000,1.000,5,1.000,1.000,1.000,5,1.000,1.000,1.000
1,NER_SG,Rijksoverheid,1.000,1.000,1.000,8,1.000,1.000,1.000,10,0.846,0.846,0.846,13,1.000,0.923,0.960,13,1.000,1.000,1.000,3,0.957,0.936,0.946
2,NER_SG,ANVS,0.737,0.824,0.778,17,0.000,0.000,0.000,0,1.000,1.000,1.000,21,0.955,1.000,0.977,21,0.000,0.000,0.000,0,0.903,0.949,0.926
3,NER_SG,DNB,1.000,1.000,1.000,5,1.000,1.000,1.000,11,1.000,0.833,0.909,6,1.000,0.833,0.909,6,0.600,0.750,0.667,4,0.935,0.906,0.921
4,NER_SG,Provincie Drenthe,1.000,1.000,1.000,7,1.000,0.900,0.947,10,0.900,0.900,0.900,10,0.900,0.900,0.900,10,0.000,0.000,0.000,1,0.944,0.895,0.919
5,NER_SG,ACM,0.750,0.923,0.828,13,0.679,0.792,0.731,24,0.938,0.789,0.857,19,1.000,0.789,0.882,19,0.765,1.000,0.867,13,0.804,0.841,0.822
6,NER_SG,KSA,0.600,0.750,0.667,4,0.667,0.857,0.750,7,1.000,0.750,0.857,8,0.857,0.750,0.800,8,1.000,1.000,1.000,6,0.818,0.818,0.818


...

,Model,Entity,P,R,F1,#
0,silver_only_pred,Recipient,0.950,0.322,0.481,59
1,silver_only_pred,Authority,0.984,0.910,0.946,67
2,silver_only_pred,Legal Object,0.986,0.854,0.915,82
3,silver_only_pred,Legal Action,0.962,0.915,0.938,82
4,silver_only_pred,Legal Basis,1.000,0.406,0.578,32
5,silver_only_pred,Micro (all entities),0.975,0.739,0.841,


,Model,Authority,Recipient P,Recipient R,Recipient F1,Recipient #,Authority P,Authority R,Authority F1,Authority #,Legal Object P,Legal Object R,Legal Object F1,Legal Object #,Legal Action P,Legal Action R,Legal Action F1,Legal Action #,Legal Basis P,Legal Basis R,Legal Basis F1,Legal Basis #,Micro-P,Micro-R,Micro-F1
0,silver_to_gold_pred,Rijksoverheid,1.000,1.000,1.000,8,1.000,1.000,1.000,10,1.000,1.000,1.000,13,1.000,1.000,1.000,13,1.000,1.000,1.000,3,1.000,1.000,1.000
1,silver_to_gold_pred,Gemeente Rotterdam,1.000,1.000,1.000,5,1.000,1.000,1.000,5,1.000,1.000,1.000,5,1.000,1.000,1.000,5,1.000,1.000,1.000,5,1.000,1.000,1.000
2,silver_to_gold_pred,DNB,1.000,1.000,1.000,5,1.000,1.000,1.000,11,1.000,1.000,1.000,6,1.000,1.000,1.000,6,1.000,1.000,1.000,4,1.000,1.000,1.000
3,silver_to_gold_pred,ANVS,1.000,0.941,0.970,17,0.000,0.000,0.000,0,1.000,1.000,1.000,21,1.000,1.000,1.000,21,0.000,0.000,0.000,0,1.000,0.983,0.991
4,silver_to_gold_pred,KSA,1.000,1.000,1.000,4,1.000,0.857,0.923,7,1.000,1.000,1.000,8,1.000,1.000,1.000,8,1.000,1.000,1.000,6,1.000,0.970,0.985
5,silver_to_gold_pred,ACM,1.000,1.000,1.000,13,1.000,0.958,0.979,24,1.000,0.947,0.973,19,1.000,1.000,1.000,19,1.000,0.923,0.960,13,1.000,0.966,0.983
6,silver_to_gold_pred,Provincie Drenthe,1.000,1.000,1.000,7,1.000,1.000,1.000,10,0.900,0.900,0.900,10,1.000,1.000,1.000,10,0.000,0.000,0.000,1,0.973,0.947,0.960


,Model,Entity,P,R,F1,#
0,silver_to_gold_pred,Recipient,1.000,0.983,0.991,59
1,silver_to_gold_pred,Authority,1.000,0.970,0.985,67
2,silver_to_gold_pred,Legal Object,0.988,0.976,0.982,82
3,silver_to_gold_pred,Legal Action,1.000,1.000,1.000,82
4,silver_to_gold_pred,Legal Basis,1.000,0.938,0.968,32
5,silver_to_gold_pred,Micro (all entities),0.997,0.978,0.987,


,Model,Authority,Recipient P,Recipient R,Recipient F1,Recipient #,Authority P,Authority R,Authority F1,Authority #,Legal Object P,Legal Object R,Legal Object F1,Legal Object #,Legal Action P,Legal Action R,Legal Action F1,Legal Action #,Legal Basis P,Legal Basis R,Legal Basis F1,Legal Basis #,Micro-P,Micro-R,Micro-F1
0,NER_G,Gemeente Rotterdam,1.000,1.000,1.000,5,1.000,1.000,1.000,5,1.000,1.000,1.000,5,1.000,1.000,1.000,5,1.000,1.000,1.000,5,1.000,1.000,1.000
1,NER_G,Rijksoverheid,1.000,0.875,0.933,8,0.909,1.000,0.952,10,0.917,0.846,0.880,13,1.000,0.923,0.960,13,0.750,1.000,0.857,3,0.935,0.915,0.925
2,NER_G,DNB,0.833,1.000,0.909,5,1.000,0.909,0.952,11,1.000,0.833,0.909,6,0.833,0.833,0.833,6,1.000,1.000,1.000,4,0.935,0.906,0.921
3,NER_G,Provincie Drenthe,1.000,1.000,1.000,7,1.000,0.900,0.947,10,0.875,0.700,0.778,10,0.900,0.900,0.900,10,0.000,0.000,0.000,1,0.941,0.842,0.889
4,NER_G,ANVS,0.579,0.647,0.611,17,0.000,0.000,0.000,0,1.000,1.000,1.000,21,0.955,1.000,0.977,21,0.000,0.000,0.000,0,0.841,0.898,0.869
5,NER_G,KSA,0.667,1.000,0.800,4,0.556,0.714,0.625,7,1.000,0.750,0.857,8,1.000,0.750,0.857,8,1.000,1.000,1.000,6,0.818,0.818,0.818
6,NER_G,ACM,0.750,0.923,0.828,13,0.700,0.875,0.778,24,1.000,0.789,0.882,19,1.000,0.789,0.882,19,0.579,0.846,0.688,13,0.779,0.841,0.809
7,NER_S,Gemeente Rotterdam,0.500,1.000,0.667,5,0.833,1.000,0.909,5,0.833,1.000,0.909,5,1.000,1.000,1.000,5,1.000,0.800,0.889,5,0.774,0.960,0.857
8,NER_S,Provincie Drenthe,1.000,1.000,1.000,7,0.909,1.000,0.952,10,0.500,0.900,0.643,10,0.474,0.900,0.621,10,0.000,0.000,0.000,1,0.636,0.921,0.753
9,NER_S,Rijksoverheid,0.583,0.875,0.700,8,0.833,1.000,0.909,10,0.647,0.846,0.733,13,0.182,0.154,0.167,13,0.000,0.000,0.000,3,0.566,0.638,0.600


,Model,Entity,P,R,F1,#
0,NER_G,Recipient,0.773,0.864,0.816,59
1,NER_G,Authority,0.800,0.896,0.845,67
2,NER_G,Legal Object,0.972,0.854,0.909,82
3,NER_G,Legal Action,0.961,0.890,0.924,82
4,NER_G,Legal Basis,0.763,0.906,0.829,32
5,NER_G,Micro (all entities),0.865,0.879,0.872,
6,NER_S,Recipient,0.215,0.339,0.263,59
7,NER_S,Authority,0.583,0.836,0.687,67
8,NER_S,Legal Object,0.427,0.854,0.569,82
9,NER_S,Legal Action,0.410,0.610,0.490,82
